# Previsão de Diabetes Gestacional com XGBoost
Este notebook treina um modelo de classificação para prever se uma paciente tem diabetes gestacional.

## Importação das Bibliotecas
Todas as bibliotecas necessárias para o projeto são importadas aqui, de uma só vez.

In [4]:
# Biblioteca para manipulação de dados em formato de tabela
import pandas as pd

# Função para dividir os dados em treino e validação
from sklearn.model_selection import train_test_split

# Funções para medir o desempenho do modelo
from sklearn.metrics import accuracy_score, confusion_matrix

# Biblioteca do algoritmo XGBoost para classificação
import xgboost as xgb

# Biblioteca para salvar e carregar o modelo treinado
import pickle

# Biblioteca para suprimir avisos desnecessários na tela
import warnings

# Desativa todos os avisos para manter a saída limpa
warnings.filterwarnings('ignore')

## Etapa 1 — Ingestão dos Dados
Carregamos o arquivo CSV e separamos a variável alvo (`diabetes`) das variáveis preditoras.

In [5]:
# Carrega o arquivo CSV usando ponto e vírgula como separador de colunas
df = pd.read_csv('diabetes-gestacional.csv', sep=';')

# Remove a coluna de ID, pois ela não ajuda o modelo a aprender
df = df.drop(columns=['ID'])

# Separa a variável alvo: o que o modelo vai aprender a prever (0 = sem diabetes, 1 = com diabetes)
y = df['diabetes']

# Separa as variáveis preditoras: as informações usadas para fazer a previsão
X = df.drop(columns=['diabetes'])

# Exibe as primeiras linhas para confirmar que os dados foram carregados corretamente
print('Primeiras linhas dos dados:')
print(X.head())
print(f'\nTotal de pacientes: {len(df)}')
print(f'Variáveis preditoras: {X.columns.tolist()}')
print(f'\nDistribuição da variável alvo:')
print(y.value_counts())

Primeiras linhas dos dados:
   gravidez  glicose  pressao-arterial  espessura-triceps  insulina  \
0         0      171                80                 34        23   
1         8       92                93                 47        36   
2         7      115                47                 52        35   
3         9      103                78                 25       304   
4         1       85                59                 27        35   

   indice-massa-corporal  diabetes-pedigree  idade  
0              43.509726           1.213191     21  
1              21.240576           0.158365     23  
2              41.511523           0.079019     23  
3              29.582192           1.282870     43  
4              42.604536           0.549542     22  

Total de pacientes: 10000
Variáveis preditoras: ['gravidez', 'glicose', 'pressao-arterial', 'espessura-triceps', 'insulina', 'indice-massa-corporal', 'diabetes-pedigree', 'idade']

Distribuição da variável alvo:
diabetes
0    

## Etapa 2 — Separação dos Dados
Dividimos os dados em dois conjuntos: **treino** (80%) e **validação** (20%). 
- `random_state=42` garante que o sorteio seja sempre o mesmo, tornando o resultado reproduzível.
- `stratify=y` garante que a proporção entre pacientes com e sem diabetes seja a mesma nos dois conjuntos.

In [6]:
# Divide os dados: 80% para treino e 20% para validação
# random_state=42 garante que o sorteio se repita toda vez que o código for executado
# stratify=y mantém a mesma proporção de pacientes com e sem diabetes nas duas partes
X_treino, X_val, y_treino, y_val = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Exibe quantas pacientes ficaram em cada conjunto
print(f'Pacientes para treino:    {len(X_treino)}')
print(f'Pacientes para validação: {len(X_val)}')
print(f'\nProporção de diabetes no treino:')
print(y_treino.value_counts(normalize=True).round(3))
print(f'\nProporção de diabetes na validação:')
print(y_val.value_counts(normalize=True).round(3))

Pacientes para treino:    8000
Pacientes para validação: 2000

Proporção de diabetes no treino:
diabetes
0    0.666
1    0.334
Name: proportion, dtype: float64

Proporção de diabetes na validação:
diabetes
0    0.666
1    0.334
Name: proportion, dtype: float64


## Etapa 3 — Treinamento do Algoritmo XGBoost
Criamos e treinamos o modelo usando o algoritmo **XGBoost** (Extreme Gradient Boosting), um dos mais poderosos para classificação.

In [7]:
# Cria o modelo XGBoost com configurações que evitam avisos na tela
# random_state=42 torna o treinamento reproduzível
# eval_metric='logloss' define a métrica de avaliação usada internamente pelo XGBoost
# verbosity=0 desativa mensagens de progresso do XGBoost
modelo = xgb.XGBClassifier(
    random_state=42,
    eval_metric='logloss',
    verbosity=0
)

# Treina o modelo com os dados de treino
modelo.fit(X_treino, y_treino)

# Confirma que o treinamento foi concluído
print('Modelo XGBoost treinado com sucesso!')

Modelo XGBoost treinado com sucesso!


## Etapa 4 — Validação do Resultado
Avaliamos o desempenho do modelo nos dados de validação, que **não foram usados** durante o treinamento.

In [8]:
# Usa o modelo treinado para fazer previsões nas pacientes de validação
y_pred = modelo.predict(X_val)

# Calcula a acurácia: percentual de previsões corretas
acuracia = accuracy_score(y_val, y_pred)

# Exibe a acurácia em formato de porcentagem
print(f'Acurácia do modelo: {acuracia:.2%}')

# Calcula a matriz de confusão comparando o previsto com o real
matriz = confusion_matrix(y_val, y_pred)

# Exibe a matriz de confusão
print('\nMatriz de Confusão:')
print('                  Previsto: 0    Previsto: 1')
print(f'  Real: 0 (sem diabetes)     {matriz[0][0]:>5}          {matriz[0][1]:>5}')
print(f'  Real: 1 (com diabetes)     {matriz[1][0]:>5}          {matriz[1][1]:>5}')

# Extrai cada célula da matriz para facilitar a leitura
vn = matriz[0][0]  # Verdadeiro Negativo
fp = matriz[0][1]  # Falso Positivo
fn = matriz[1][0]  # Falso Negativo
vp = matriz[1][1]  # Verdadeiro Positivo

# Exibe a explicação de cada número da matriz
print('\nO que cada número significa:')
print(f'  {vn:>4} Verdadeiros Negativos (VN): pacientes SEM diabetes corretamente identificadas como SEM diabetes.')
print(f'  {fp:>4} Falsos Positivos      (FP): pacientes SEM diabetes incorretamente identificadas como COM diabetes.')
print(f'  {fn:>4} Falsos Negativos      (FN): pacientes COM diabetes incorretamente identificadas como SEM diabetes.')
print(f'  {vp:>4} Verdadeiros Positivos (VP): pacientes COM diabetes corretamente identificadas como COM diabetes.')
print(f'\nAtenção: os Falsos Negativos ({fn}) são os casos mais críticos,')
print('pois representam pacientes doentes que o modelo deixou escapar.')

Acurácia do modelo: 95.80%

Matriz de Confusão:
                  Previsto: 0    Previsto: 1
  Real: 0 (sem diabetes)      1294             37
  Real: 1 (com diabetes)        47            622

O que cada número significa:
  1294 Verdadeiros Negativos (VN): pacientes SEM diabetes corretamente identificadas como SEM diabetes.
    37 Falsos Positivos      (FP): pacientes SEM diabetes incorretamente identificadas como COM diabetes.
    47 Falsos Negativos      (FN): pacientes COM diabetes incorretamente identificadas como SEM diabetes.
   622 Verdadeiros Positivos (VP): pacientes COM diabetes corretamente identificadas como COM diabetes.

Atenção: os Falsos Negativos (47) são os casos mais críticos,
pois representam pacientes doentes que o modelo deixou escapar.


## Salvamento do Modelo Treinado
Exportamos o modelo para um arquivo `.pkl` para que ele possa ser carregado e usado em uma interface gráfica sem precisar treinar novamente.

In [9]:
# Define o nome do arquivo onde o modelo será salvo
nome_arquivo = 'preditor-diabetes.pkl'

# Abre o arquivo em modo de escrita binária e salva o modelo dentro dele
with open(nome_arquivo, 'wb') as arquivo:
    pickle.dump(modelo, arquivo)

# Confirma que o arquivo foi salvo com sucesso
print(f'Modelo salvo com sucesso no arquivo: {nome_arquivo}')
print(f'\nPara usar o modelo em outra aplicação, carregue-o assim:')
print(f"  with open('{nome_arquivo}', 'rb') as f:")
print(f"      modelo_carregado = pickle.load(f)")
print(f"  previsao = modelo_carregado.predict(novos_dados)")

Modelo salvo com sucesso no arquivo: preditor-diabetes.pkl

Para usar o modelo em outra aplicação, carregue-o assim:
  with open('preditor-diabetes.pkl', 'rb') as f:
      modelo_carregado = pickle.load(f)
  previsao = modelo_carregado.predict(novos_dados)
